# Основы Pandas для начинающих

https://stepik.org/lesson/1659548/step/1?unit=1682395

<details>
    <summary>Таблица locations - местоположения филиалов:</summary>
    
* id - уникальный номер локации
* region - название округа
* city - название города
</details>

<details>
    <summary>Таблица managers - управляющие филиалами:</summary>

* id - уникальный номер менеджера
* fio - ФИО
* sex - Пол
* join_year - год трудоустройства в компанию
* join_month - месяц трудоустройства в компанию
* join_day - день трудоустройства в компанию
</details>


<details>
    <summary>Таблица branches - филиалы:</summary>

* branch_id - уникальный номер филиала
* location_id - уникальный номер местонахождения
* store_area - площадь склада
* open_date - дата открытия филиала
* manager_id - уникальный номер менеджера
</details>


<details>
    <summary>Таблица reports - отчетность по филиалам:</summary>

* branch_id - уникальный номер филиала
* report_year - год отчета
* type - название показателя
  >(sales - общий объем продаж (руб.),   
  num_transactions - количество продаж,   
  num_customers - уникальное количество покупателей,   
  discount_rate - средняя * процентная скидка применяемая в филиале,   
  expenses - операционные расходы за период)   
* value - значение показателя
</details>

In [1]:
#!pip install mariadb
import sys
import requests
import json
from datetime import datetime, date, timedelta

import mariadb
import pandas as pd
import numpy as np

# Параметры подключения к базе данных
config = {
    'user': 'corp_reader',         # имя пользователя
    'password': 'corp_reader',     # пароль
    'host': '149.154.71.99',             # адрес сервера
    'port': 3306,                    # порт MariaDB
    'database': 'corp'      # название базы данных
}

def read_maria_db(query, config):
    # Подключение к MariaDB
    try:
        conn = mariadb.connect(**config)
    except mariadb.Error as e:
        print(f"Ошибка подключения к MariaDB: {e}")
        sys.exit(1)
    # Чтение данных в DataFrame
    df = pd.read_sql(query, conn)
    # Закрытие соединения
    conn.close()

    return df

## Задание #1

https://stepik.org/lesson/1659548/step/2?unit=1682395

In [65]:
query = """
SELECT
    b.branch_id
    , m.fio
FROM managers m
INNER JOIN branches b ON m.id = b.manager_id
"""

df_1 = read_maria_db(query, config)

C:\Users\ilyao\AppData\Local\Temp\ipykernel_6640\538137961.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [74]:
def get_short_full_name(full_name:str):
    list_name = full_name.split()
    if len(list_name) == 3:
        last_name, first_name, middle_name = list_name
        return f"{last_name} {first_name[0]}.{middle_name[0]}."
    # if len(list_name) == 2:
    #     last_name, first_name = list_name
    #     return f"{last_name} {first_name[0]}."
    return full_name

(df_1
 .assign(manager_short_fio = lambda df: df.fio.apply(get_short_full_name))
 .sort_values('branch_id')
 .loc[:, ['branch_id','manager_short_fio']]
 .to_csv('pandas_for_dummies_apply.csv', index=False, sep=';')
)

## Задание #2

https://stepik.org/lesson/1659548/step/3?unit=1682395

In [2]:
query = """
SELECT *
FROM branches b
"""

df = read_maria_db(query, config)

C:\Users\ilyao\AppData\Local\Temp\ipykernel_11048\4186592254.py:24: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [3]:
today = datetime(2025,3,5)

(df
 .assign(open_date = pd.to_datetime(df.open_date)
        , store_age = lambda df: df.open_date.apply(lambda x: (today-x).days//365)
        , store_age2 = lambda df: df.open_date.apply(lambda x: (today.year-x.year)-1 if (today.month, today.day) < (x.month, x.day) else (today.year-x.year))
 )
 .loc[:, ['branch_id','store_age']]
 .to_csv('pandas_for_dummies_apply.csv', index=False, sep=';')
)

AttributeError: 'DataFrame' object has no attribute 'open_date'

## Задание #3

https://stepik.org/lesson/1659548/step/4?unit=1682395

In [126]:
query = """
SELECT
    branch_id
    , concat(type, '_', report_year) as discount_rate_year
    , value
FROM reports r
WHERE type = 'discount_rate'
"""

df = read_maria_db(query, config)

C:\Users\ilyao\AppData\Local\Temp\ipykernel_16536\2423696836.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [127]:
df.head()

,branch_id,discount_rate_year,value
0,100,discount_rate_2020,2
1,100,discount_rate_2021,3
2,100,discount_rate_2022,0
3,100,discount_rate_2023,13
4,100,discount_rate_2024,12


In [78]:
(df
 .pivot(columns='discount_rate_year', index='branch_id', values='value')
 .reset_index()
 .assign(max_discount_year = lambda df: 
         df.iloc[:, list(range(1,6))].apply(
             lambda y: max(zip(y, range(2020, 2025))
                           , key=lambda x: x[0])[1], axis=1)
        )
 .loc[:, ['branch_id','max_discount_year']]
 .to_csv('pandas_for_dummies_apply.csv', index=False, sep=';')
 # .head()
)

## Задание #4

https://stepik.org/lesson/1659548/step/5?unit=1682395

In [102]:
query = """
SELECT
    branch_id
    , concat(type, '_', report_year) as type_year
    , value
FROM reports r
WHERE type in ('expenses', 'sales')
ORDER BY type DESC, report_year
"""

df = read_maria_db(query, config)

C:\Users\ilyao\AppData\Local\Temp\ipykernel_11048\4186592254.py:24: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [112]:
expenses_list = ['expenses_'+str(y) for y in range(2020, 2025)]
sales_list = ['sales_'+str(y) for y in range(2020, 2025)]

In [152]:
(df
 .pivot(columns='type_year', index='branch_id', values='value')
 .reset_index()
 .assign(payback_year = lambda df: df.apply(
                             lambda row: min(
                                 [i[2] for i in zip(expenses_list, sales_list, range(2020, 2025)) if row[i[1]]*0.25 > row[i[0]]]
                             )
                             , axis=1
                                           )
         )
 .loc[:, ['branch_id','payback_year']]
 .to_csv('pandas_for_dummies_apply.csv', index=False, sep=';')
 # .head()
)

## Задание #5

https://stepik.org/lesson/1659548/step/6?unit=1682395

In [89]:
query = """
SELECT
    branch_id
    , concat(type, '_', report_year) as type_year
    , value
FROM reports r
WHERE type in ('expenses', 'sales', 'discount_rate','num_transactions')
ORDER BY type DESC, report_year
"""

df1 = read_maria_db(query, config)

C:\Users\ilyao\AppData\Local\Temp\ipykernel_16536\2423696836.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [90]:
discount_cols = ['discount_rate_' + str(year) for year in range(2020, 2024+1)]
sales__cols = ['sales_' + str(year) for year in range(2020, 2024+1)]
expenses_cols = ['expenses_' + str(year) for year in range(2020, 2024+1)]
transactions_cols = ['num_transactions_' + str(year) for year in range(2020, 2024+1)]

(df1
 .pivot(columns='type_year', index='branch_id', values='value')
 .reset_index()
 .assign(mean_discount = lambda df: df.loc[:, discount_cols].apply(np.mean, axis=1)
         , sum_sales = lambda df: df.loc[:, sales__cols].apply(np.sum, axis=1)
         , sum_expenses = lambda df: df.loc[:, expenses_cols].apply(np.sum, axis=1)
         , sum_transactions = lambda df: df.loc[:, transactions_cols].apply(np.sum, axis=1)
        )
 .drop(columns=[measure+str(year) for measure in ['discount_rate_','expenses_','sales_', 'num_transactions_'] for year in range(2020, 2024+1)])
 .assign(roi = lambda df: df.sum_sales.div(np.where(df.sum_expenses==0, np.nan, df.sum_expenses))
        , performance_score = lambda df: df.apply(lambda row: round(row.roi * np.log(1+row.sum_transactions) * row.mean_discount, 5), axis=1))
 # .loc[:, ['branch_id','performance_score']]
 .sort_values('branch_id')
 # .to_csv('pandas_for_dummies_apply.csv', index=False, sep=';')
 .head(8)
)

type_year,branch_id,mean_discount,sum_sales,sum_expenses,sum_transactions,roi,performance_score
0,100,6.0,219326094,26541713,1031,8.263449,344.05303
1,101,6.4,337181987,28808523,1724,11.704244,558.28176
2,102,4.6,306777461,32938352,1214,9.313686,304.29206
3,103,7.4,218519270,28623638,964,7.634224,388.22891
4,104,6.0,250624721,30737138,1621,8.153808,361.60908
5,105,6.8,205323516,17480599,1347,11.745794,575.58344
6,106,5.8,233593157,29781682,1170,7.843518,321.43174
7,107,9.4,310181309,31137588,1321,9.961636,672.97693


In [94]:
query = """
SELECT
    branch_id
    , avg(if(type='discount_rate', value, null)) as mean_discount
    , sum(if(type='sales', value, null)) as sum_sales
    , sum(if(type='expenses', value, null)) as sum_expenses
    , sum(if(type='num_transactions', value, null)) as sum_transactions
    , avg(if(type='sales', value, null)) / nullif(avg(if(type='expenses', value, null)), 0) as roi -- При сумме происходит округление до 4 знаков
FROM reports r
WHERE type in ('expenses', 'sales', 'discount_rate','num_transactions')
group by 1
order by branch_id
;
"""

df = read_maria_db(query, config)

C:\Users\ilyao\AppData\Local\Temp\ipykernel_16536\2423696836.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [96]:
(df
 .assign(performance_score = lambda df: df.apply(lambda row: round(row.roi * np.log(1+row.sum_transactions) * row.mean_discount, 2), axis=1))
 .loc[:, ['branch_id','performance_score']]
 .to_csv('pandas_for_dummies_apply.csv', index=False, sep=';')
 # .head(8)
)

## Задание #6

https://stepik.org/lesson/1659548/step/7?unit=1682395

In [133]:
query = """
SELECT
    branch_id
    , report_year
    , sum(value) as sales
FROM reports r
WHERE type = 'sales'
group by 1,2
order by branch_id
;
"""

df = read_maria_db(query, config)

C:\Users\ilyao\AppData\Local\Temp\ipykernel_16536\2423696836.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [205]:
(df
 .pivot(columns='report_year',index='branch_id', values='sales')
 .reset_index()
 .rename_axis(None, axis=1)
 .assign(yoy_growth_pattern = lambda df: df.apply(lambda row: sum([1 if (row[year]/row[year-1]-1)>0 else -1 for year in range(2021, 2025)]), axis=1))
 .assign(yoy_growth_pattern = lambda df: np.where(df.yoy_growth_pattern==4, 'consistent growth'
                                                      , np.where(df.yoy_growth_pattern==-4, 'consistent decline', 'volatile'))
        )
 .loc[:, ['branch_id','yoy_growth_pattern']]
 .sort_values('branch_id')
 .to_csv('pandas_for_dummies_apply.csv', index=False, sep=';')
 # .head()
)

## Задание #7
Для каждого филиала вычислите маржу (profit margin) за каждый год 2020–2024   

https://stepik.org/lesson/1659548/step/8?unit=1682395

In [5]:
query = """
SELECT
    branch_id
    , report_year
    , sum(if(type='sales', value, null)) as sum_sales
    , sum(if(type='expenses', value, null)) as sum_expenses
FROM reports r
WHERE type in ('expenses', 'sales')
group by 1,2
order by branch_id
;
"""

df = read_maria_db(query, config)

C:\Users\ilyao\AppData\Local\Temp\ipykernel_16108\2423696836.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [33]:
(df
 .assign(fm_perc = lambda df: 1-df.sum_expenses / df.sum_sales
         # , fm_perc = lambda df: df.apply(lambda row: 1 - row.sum_expenses / row.sum_sales, axis=1)
        )
 .groupby('branch_id', as_index=False)
     # .fm_perc.std(ddof=0).round(3).rename(columns={'fm_perc':'profit_margin_volatility'})
     .agg(profit_margin_volatility = ('fm_perc', lambda x: np.std(x, ddof=0).round(3)))
 .to_csv('pandas_for_dummies_apply.csv', index=False, sep=';')
 # .head()
)

## Задание #8
Оценка «плотности» транзакций на клиента (transaction_density) и классификация

https://stepik.org/lesson/1659548/step/9?unit=1682395

In [43]:
query = """
SELECT
    branch_id
    , report_year
    , sum(if(type='num_transactions', value, null)) as sum_transactions
    , sum(if(type='num_customers', value, null)) as sum_customers
FROM reports r
WHERE type in ('num_transactions', 'num_customers')
group by 1,2
order by branch_id
;
"""

df = read_maria_db(query, config)

C:\Users\ilyao\AppData\Local\Temp\ipykernel_16108\2423696836.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [67]:
def transaction_density(row):
    customer_cols = ['cst_'+str(y) for y in range(2020, 2025)]
    transactions_cols = ['trn_'+str(y) for y in range(2020, 2025)]
    ratio = []
    for cst_yr, trn_yr in zip(customer_cols, transactions_cols):
        ratio.append(row[trn_yr] / row[cst_yr])
    avg_ratio = sum(ratio)/len(ratio)
    if avg_ratio >= 5:
        return "High retention"
    elif avg_ratio < 3:
        return "Low retention"
    else:
        return "Moderate retention"

(df
 .pivot(columns='report_year',index='branch_id', values='sum_customers')
 .rename(columns={y:'cst_'+str(y) for y in range(2020, 2025)})
 .join(df
       .pivot(columns='report_year',index='branch_id', values='sum_transactions')
       .rename(columns={y:'trn_'+str(y) for y in range(2020, 2025)})
      )
 .reset_index().rename_axis(None, axis=1)
 .assign(customer_retention_category = lambda df: df.apply(transaction_density, axis=1))
 .loc[:, ['branch_id','customer_retention_category']]
 .to_csv('pandas_for_dummies_apply.csv', index=False, sep=';')
 # .head()
)

In [34]:
query = """
SELECT
    branch_id
    , report_year
    , sum(if(type='num_transactions', value, null)) as sum_transactions
    , sum(if(type='num_customers', value, null)) as sum_customers
FROM reports r
WHERE type in ('num_transactions', 'num_customers')
group by 1,2
order by branch_id
;
"""

df = read_maria_db(query, config)

C:\Users\ilyao\AppData\Local\Temp\ipykernel_16108\2423696836.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [42]:
(df
 .assign(ratio = lambda df: df.sum_transactions / df.sum_customers)
 .groupby('branch_id', as_index=False)
     .agg(avg_ratio = ('ratio','mean'))
 .assign(customer_retention_category = lambda df: np.where(df.avg_ratio>=5, 'High retention', np.where(df.avg_ratio<3, 'Low retention', 'Moderate retention')))
 .loc[:, ['branch_id','customer_retention_category']]
 .to_csv('pandas_for_dummies_apply.csv', index=False, sep=';')
 # .head()
)

## Задание #9
Анализ влияния прихода менеджера на продажи

https://stepik.org/lesson/1659548/step/10?unit=1682395

In [49]:
query = """
select
    r.branch_id
    , b.manager_id
--    , r.report_year
    , case
        when r.report_year < m.join_year then 'pre-join'
        else 'post-join'
    end as join_period
    , avg(if(type='sales', value, null)) as avr_sales
from reports r
inner join branches b on b.branch_id = r.branch_id
left join managers m on m.id = b.manager_id 
    -- and m.join_year = r.report_year
where r.type = 'sales'
group by 1,2,3
order by r.branch_id, r.report_year
;
"""

df = read_maria_db(query, config)

C:\Users\ilyao\AppData\Local\Temp\ipykernel_6340\2423696836.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [57]:
def manager_impact_class(row):
    if row['pre-join'] == 0 or row['post-join'] == 0:
        return 'no_data'
    ratio = row['post-join'] / row['pre-join']
    if ratio > 1.2:
        return 'improved'
    elif 0.8 <= ratio <= 1.2:
        return 'stable'
    elif ratio < 0.8:
        return 'worse'
    else:
        return 'no_data'

# df.info(verbose=False)
(df
 .pivot(columns='join_period',index=['branch_id','manager_id'], values='avr_sales')
 .reset_index().rename_axis(None, axis=1)
 .fillna({'post-join':0, 'pre-join':0})
 .assign(manager_impact_class = lambda df:
        df.apply(manager_impact_class, axis=1)
        )
 .loc[:, ['branch_id','manager_impact_class']]
 .to_csv('pandas_for_dummies_apply.csv', index=False, sep=';')
 # .head(5)
)

## Задание #10
Определение «аномальных» лет по марже

https://stepik.org/lesson/1659548/step/11?unit=1682395

In [2]:
query = """
select
    branch_id
    , report_year
    , sum(if(type='sales', value, null)) as sales
    , sum(if(type='expenses', value, null)) as expenses
from reports r
where type in ('expenses', 'sales')
group by 1,2
order by branch_id
;
"""

df = read_maria_db(query, config)

C:\Users\ilyao\AppData\Local\Temp\ipykernel_19040\2423696836.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [78]:
def margin_outlier_years(row, cols, year_start, year_end, COEFF=1.3):
    # years_fm = {y: for c, y in zip(cols, range(year_start, year_end+1))}
    cols = [[f"{i}_{j}" for i in cols] for j in range(year_start, year_end+1)]
    margine = np.array([1-row[e]/row[s] for e,s in cols])
    fm_mean, fm_std = margine.mean(), margine.std(ddof=0)
    anom_year = np.array(range(year_start, year_end+1))[abs(margine-fm_mean) > (COEFF * fm_std)]
    return ', '.join(map(str, anom_year)) if any(anom_year) else 'none'


(df
 # подготавливаем данные
 .assign(report_year = lambda df: df.report_year.astype(str))
 .pivot_table(values=['sales','expenses'], index='branch_id', columns='report_year', aggfunc='sum')
 .pipe(lambda df: df.set_axis(df.columns.map('_'.join), axis=1))
     # .pipe(lambda x: x.set_axis([f'{col[0]}_{col[1]}' for col in x.columns], axis=1))
 .reset_index()
 # решение через apply
 .assign(margin_outlier_years = lambda df: 
         df.apply(margin_outlier_years, axis=1, args=(['expenses','sales'], 2020, 2024))
        )
 .loc[:, ['branch_id','margin_outlier_years']]
 # .to_csv('pandas_for_dummies_apply.csv', index=False, sep=';')
 .head()
)

,branch_id,margin_outlier_years
0,100,2023
1,101,"2023, 2024"
2,102,2020
3,103,2024
4,104,2023


## Задание #11
Вычислите операционную эффективность (operational_efficiency)  с учетом региона, города и площади

https://stepik.org/lesson/1659548/step/12?unit=1682395

In [2]:
region_factor = {
    'Сибирский федеральный округ': 1.5,
    'Южный федеральный округ': 1.3,
    'Уральский федеральный округ': 1.4,
    'Центральный федеральный округ': 1.1,
    'Приволжский федеральный округ': 1.2,
    'Северо-Западный федеральный округ': 1.2
}

city_factor = {
    'Красноярск': 1.1,
    'Краснодар': 1.2,
    'Екатеринбург': 1.1,
    'Челябинск': 1.1,
    'Воронеж': 1.0,
    'Москва': 1.2,
    'Нижний Новгород': 1.05,
    'Казань': 1.1,
    'Самара': 1.0,
    'Ростов-на-Дону': 1.05,
    'Волгоград': 1.0,
    'Барнаул': 1.0,
    'Санкт-Петербург': 1.2,
    'Пермь': 1.0,
    'Уфа': 1.0,
    'Омск': 1.05,
    'Саратов': 1.05,
    'Тюмень': 1.1,
    'Новосибирск': 1.1
}

In [3]:
query = """
select
    r.branch_id
    , l.region
    , l.city
    , b.store_area
    , sum(value) as total_sales
from reports r
inner join branches b on b.branch_id = r.branch_id
inner join locations l on l.id = b.location_id
where r.type = 'sales'
group by 1,2,3,4
order by r.branch_id
;
"""

df = read_maria_db(query, config)

C:\Users\ilyao\AppData\Local\Temp\ipykernel_17816\2027733276.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [9]:
def efficiency_class(row:pd.DataFrame)->str:
    region_coeff = region_factor.get(row.region, 1)
    city_coeff = city_factor.get(row.city, 1)
    operational_efficiency = row.total_sales / (row.store_area*region_coeff*city_coeff)
    return 'High' if operational_efficiency > 1e6 else 'Low' if operational_efficiency < 3e5 else 'Medium'

(df
 .assign(efficiency_class = lambda df:
        df.apply(efficiency_class, axis=1)
        )
 .loc[:, ['branch_id','efficiency_class']]
 # .to_csv('pandas_for_dummies_apply.csv', index=False, sep=';')
 .head()
)

,branch_id,efficiency_class
0,100,Low
1,101,Medium
2,102,High
3,103,Low
4,104,Medium


## Задание #12
Для каждого филиала найдите рассчитайте корреляцию между ростом скидок и ростом продаж

https://stepik.org/lesson/1659548/step/13?unit=1682395

In [143]:
query = """
select
    branch_id
    , report_year
    , sum(if(type='sales', value, null)) as sales
    , sum(if(type='discount_rate', value, null)) as discount_rate
from reports r
where type in ('discount_rate', 'sales')
group by 1,2
order by branch_id
;
"""

df = read_maria_db(query, config)

C:\Users\ilyao\AppData\Local\Temp\ipykernel_19040\2423696836.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [172]:
def corr_category(row):
    sales_yoy = np.array([row[f'sales_{year}']-row[f'sales_{year-1}'] for year in (range(2021, 2025))])
    disc_yoy = np.array([row[f'discount_rate_{year}']-row[f'discount_rate_{year-1}'] for year in (range(2021, 2025))])
    corr = np.corrcoef(sales_yoy, disc_yoy)[0][1]
    if corr == None or (abs(sales_yoy).sum()==0 or abs(disc_yoy).sum()==0):
        return'undefined'
    return 'positive' if corr>0.5 else 'negative' if corr<-0.5 else 'neutral'

(df
 # подготавливаем данные
 .assign(report_year = lambda df: df.report_year.astype(str))
 .pivot_table(values=['sales','discount_rate'], index='branch_id', columns='report_year', aggfunc='sum')
 .pipe(lambda df: df.set_axis(df.columns.map('_'.join), axis=1))
 .reset_index()
 # решение через apply
 .assign(corr_category = lambda df: 
         df.apply(corr_category, axis=1)
        )
 .loc[:,['branch_id','corr_category']]
 .to_csv('pandas_for_dummies_apply.csv', index=False, sep=';')
 # .head()
)

## Задание #13

https://stepik.org/lesson/1659548/step/14?unit=1682395

In [177]:
query = """
select
    branch_id
    , report_year
    , sum(value) as sales
from reports r
where type = 'sales'
group by 1,2
order by branch_id
;
"""

df = read_maria_db(query, config)

C:\Users\ilyao\AppData\Local\Temp\ipykernel_19040\2027733276.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [248]:
def forecast_sales(row, cols):
    # Пример запроса к сервису прогнозирования продаж
    url = "http://149.154.71.99:8000/forecast"
    payload = json.dumps({
      "sales": [
        row[cols[0]],
        row[cols[1]],
        row[cols[2]],
        row[cols[3]]
      ]
    })
    headers = {
      'Content-Type': 'application/json',
      'x-api-key': 'mdQ0UKnMPpydKXl'
    }
    response = requests.request("POST", url, headers=headers, data=payload)
    return json.loads(response.text)['future_sales']

(df
 .assign(report_year = lambda df: df.report_year.astype(str))
 .pivot(values='sales', index='branch_id', columns='report_year')
 .reset_index()
 .rename_axis(None, axis=1)
 .rename(columns={str(i):f"sales_{i}" for i in range(2020, 2025)})
 .assign(sales_forecast_2025 = lambda df: df.apply(forecast_sales, axis=1, args=[['sales_2021','sales_2022','sales_2023','sales_2024']])
         , sales_forecast_2026 = lambda df: df.apply(forecast_sales, axis=1, args=[['sales_2022','sales_2023','sales_2024','sales_forecast_2025']])
         , sales_forecast_2027 = lambda df: df.apply(forecast_sales, axis=1, args=[['sales_2023','sales_2024','sales_forecast_2025','sales_forecast_2026']])
        )
 .loc[:, ['branch_id','sales_forecast_2025','sales_forecast_2026','sales_forecast_2027']]
 .to_csv('pandas_for_dummies_apply.csv', index=False, sep=';')
 # .head()
)

## Задание #14

In [279]:
url = "http://149.154.71.99:8000/branch_rating/103"

payload = {}
headers = {
  'x-api-key': 'mdQ0UKnMPpydKXl'
}

response = requests.request("GET", url, headers=headers, data=payload)

print(response.text)

{"ratings":[3.38,2.17,4.98,2.17,4.09,4.95,2.72,2.43,2.36,2.91,2.3,4.08]}


In [2]:
query = """
select
    branch_id
from branches
order by branch_id
;
"""

df = read_maria_db(query, config)

C:\Users\ilyao\AppData\Local\Temp\ipykernel_7092\2027733276.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [5]:
def get_rating(branch_id):
    url = f"http://149.154.71.99:8000/branch_rating/{branch_id}"
    payload = {}
    headers = {
      'x-api-key': 'mdQ0UKnMPpydKXl'
    }
    response = requests.request("GET", url, headers=headers, data=payload)
    rating = np.array(json.loads(response.text)['ratings'])

    avg_rating_12m = rating.mean().round(1)
    fraction_low_rating = (rating<3).mean().round(2)

    if avg_rating_12m >= 3.8:
        service_quality = 'Excellent'
    elif fraction_low_rating > 0.3:
        service_quality = 'Needs Improvement'
    else:
        service_quality = 'Normal'

    return (avg_rating_12m, fraction_low_rating, service_quality)
    # return (round_math(avg_rating_12m), round_math(fraction_low_rating), service_quality)

# df = \
(df
 # .head()
 .assign(rating = lambda df: df.branch_id.apply(get_rating))
 .pipe(lambda df_temp: df_temp.join(df_temp.rating.apply(pd.Series)))
 .rename(columns={0:'avg_rating_12m', 1:'low_rating_share_12m', 2:'service_quality'})
 .loc[:, ['branch_id','avg_rating_12m','low_rating_share_12m','service_quality']]
 .to_csv('pandas_for_dummies_apply.csv', index=False, sep=';')
 # .head()
)

In [353]:
(df
 # .loc[:, ['branch_id','avg_rating_12m','low_rating_share_12m','service_quality']]
 # .assign(rat = lambda df: df[3].apply(np.mean).apply(round_math))
)

,branch_id
0,100
1,101
2,102
3,103
4,104
5,105
6,106
7,107
8,108
9,109
